In [1]:
import pandas as pd
import numpy as np
import itertools
import pandas_gbq
import datetime
import matplotlib.pyplot as plt
from datetime import *
from datetime import datetime, timedelta, date
from pathlib import Path
from PIL import Image
# %load_ext google.colab.data_table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
query2=f"""
SELECT
*
FROM `perceptive-ivy-290216.f1_api.sprint_lap_time`  A
# WHERE A.Year=2024
# AND A.GP="Monaco Grand Prix"
# AND A.DRIVER='HAM'
ORDER BY LapNumber, LapStartTime
"""
track3=pandas_gbq.read_gbq(query2,project_id,dialect='standard')

Downloading: 100%|██████████|


In [4]:
track2=track3[(track3["GP"]=='Qatar Grand Prix')&(track3["Year"]==2025)]

In [5]:
track2.head()
year=track2['Year'].iloc[0]
gp=track2['GP'].iloc[0]

In [6]:
track2['LapTime']= pd.to_timedelta(track2["LapTime"])

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_38334/4173085553.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2['LapTime']= pd.to_timedelta(track2["LapTime"])


In [7]:
# track2['LapTime'].dt.total_seconds().min()*1.07
track2=track2[track2['LapNumber']!=1.0]
track2.loc[:, "LapTime (s)"] = track2["LapTime"].dt.total_seconds()

In [8]:
# quicklaps=track2[track2["LapTime"]<track2['LapTime'].min()*1.35]
quicklaps=track2
quicklaps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP,LapTime (s)
660,0 days 00:48:51.615000,PIA,81,0 days 00:01:25.086000,2.0,1.0,NaT,NaT,0 days 00:00:31.501000,0 days 00:00:28.571000,0 days 00:00:25.014000,0 days 00:47:58.065000,0 days 00:48:26.636000,0 days 00:48:51.650000,242.0,288.0,278.0,300.0,True,MEDIUM,7.0,False,McLaren,0 days 00:47:26.529000,2025-11-29 14:05:00.354,1,1.0,False,,False,True,2025,Qatar Grand Prix,85.086
663,0 days 00:48:52.888000,RUS,63,0 days 00:01:25.085000,2.0,1.0,NaT,NaT,0 days 00:00:31.518000,0 days 00:00:28.766000,0 days 00:00:24.801000,0 days 00:47:59.341000,0 days 00:48:28.107000,0 days 00:48:52.908000,NaN,289.0,280.0,293.0,True,MEDIUM,8.0,False,Mercedes,0 days 00:47:27.803000,2025-11-29 14:05:01.628,1,2.0,False,,False,True,2025,Qatar Grand Prix,85.085
667,0 days 00:48:54.111000,NOR,4,0 days 00:01:25.135000,2.0,2.0,NaT,NaT,0 days 00:00:31.619000,0 days 00:00:28.844000,0 days 00:00:24.672000,0 days 00:48:00.622000,0 days 00:48:29.466000,0 days 00:48:54.138000,241.0,288.0,280.0,305.0,True,HARD,2.0,True,McLaren,0 days 00:47:28.976000,2025-11-29 14:05:02.801,1,3.0,False,,False,True,2025,Qatar Grand Prix,85.135
670,0 days 00:48:54.729000,VER,1,0 days 00:01:25.136000,2.0,2.0,NaT,NaT,0 days 00:00:31.491000,0 days 00:00:29.106000,0 days 00:00:24.539000,0 days 00:48:01.109000,0 days 00:48:30.215000,0 days 00:48:54.754000,244.0,287.0,283.0,302.0,True,HARD,2.0,True,Red Bull Racing,0 days 00:47:29.593000,2025-11-29 14:05:03.418,1,4.0,False,,False,True,2025,Qatar Grand Prix,85.136
673,0 days 00:48:56.125000,TSU,22,0 days 00:01:25.493000,2.0,2.0,NaT,NaT,0 days 00:00:31.647000,0 days 00:00:28.978000,0 days 00:00:24.868000,0 days 00:48:02.312000,0 days 00:48:31.290000,0 days 00:48:56.158000,241.0,286.0,281.0,298.0,True,HARD,2.0,True,Red Bull Racing,0 days 00:47:30.632000,2025-11-29 14:05:04.457,1,5.0,False,,False,True,2025,Qatar Grand Prix,85.493


In [9]:
#Remove Pitstops and Track Status other than Clear to remove slow laps
quicklaps=quicklaps[((quicklaps["PitOutTime"]=='NaT')&(quicklaps["PitInTime"]=='NaT')&(~quicklaps["TrackStatus"].isin(['4','41','5','6','7','124'])))]

In [10]:
transformed_laps = quicklaps.copy()
transformed_laps.loc[:, "LapTime (s)"] = quicklaps["LapTime"].dt.total_seconds()

# order the team from the fastest (lowest median lap time) to slower
team_order = (
    transformed_laps[["Team", "LapTime (s)"]].groupby("Team").median()["LapTime (s)"].sort_values().index
)
print(team_order)

Index(['McLaren', 'Mercedes', 'Red Bull Racing', 'Aston Martin', 'Williams',
       'Racing Bulls', 'Kick Sauber', 'Ferrari', 'Haas F1 Team', 'Alpine'],
      dtype='object', name='Team')


In [11]:
transformed_laps['median'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('median')
transformed_laps.tail()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP,LapTime (s),median
8214,0 days 01:13:31.676000,HUL,27,0 days 00:01:26.522000,19.0,1.0,NaT,NaT,0 days 00:00:31.982000,0 days 00:00:29.564000,0 days 00:00:24.976000,0 days 01:12:37.161000,0 days 01:13:06.725000,0 days 01:13:31.701000,238.0,288.0,278.0,333.0,False,MEDIUM,25.0,False,Kick Sauber,0 days 01:12:05.154000,2025-11-29 14:29:38.979,1,16.0,False,,False,True,2025,Qatar Grand Prix,86.522,86.2015
8215,0 days 01:13:38.209000,HAM,44,0 days 00:01:28.912000,19.0,1.0,NaT,NaT,0 days 00:00:32.788000,0 days 00:00:29.955000,0 days 00:00:26.169000,0 days 01:12:42.113000,0 days 01:13:12.068000,0 days 01:13:38.237000,239.0,287.0,242.0,309.0,False,MEDIUM,19.0,True,Ferrari,0 days 01:12:09.297000,2025-11-29 14:29:43.122,1,17.0,False,,False,True,2025,Qatar Grand Prix,88.912,86.2170
8216,0 days 01:14:01.572000,GAS,10,0 days 00:01:23.188000,19.0,2.0,NaT,NaT,0 days 00:00:30.854000,0 days 00:00:28.154000,0 days 00:00:24.180000,0 days 01:13:09.290000,0 days 01:13:37.444000,0 days 01:14:01.624000,250.0,285.0,273.0,300.0,True,SOFT,24.0,False,Alpine,0 days 01:12:38.384000,2025-11-29 14:30:12.209,1,18.0,False,,False,True,2025,Qatar Grand Prix,83.188,86.5715
8217,0 days 01:14:09.998000,STR,18,0 days 00:01:23.809000,19.0,2.0,NaT,NaT,0 days 00:00:31.077000,0 days 00:00:28.505000,0 days 00:00:24.227000,0 days 01:13:17.286000,0 days 01:13:45.791000,0 days 01:14:10.018000,243.0,286.0,279.0,303.0,False,SOFT,16.0,False,Aston Martin,0 days 01:12:46.189000,2025-11-29 14:30:20.014,1,19.0,False,,False,True,2025,Qatar Grand Prix,83.809,85.8590
8218,0 days 01:14:12.842000,COL,43,0 days 00:01:23.765000,19.0,2.0,NaT,NaT,0 days 00:00:30.945000,0 days 00:00:28.537000,0 days 00:00:24.283000,0 days 01:13:20.084000,0 days 01:13:48.621000,0 days 01:14:12.904000,249.0,284.0,271.0,303.0,True,SOFT,8.0,False,Alpine,0 days 01:12:49.077000,2025-11-29 14:30:22.902,1,20.0,False,,False,True,2025,Qatar Grand Prix,83.765,86.5715


In [12]:
fig_pace=px.box(
    transformed_laps.sort_values(by=["median","LapNumber"]),
    x="Team",
    y="LapTime (s)",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace Plot for the {} {} Sprint Race</b>".format(year,gp),
    # palette=team_palette,
    # whiskerprops=dict(color="white"),
    # boxprops=dict(edgecolor="white"),
    # medianprops=dict(color="grey"),
    # capprops=dict(color="white"),
    height=700, width=1200,

color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Racing Bulls": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)
fig_pace.update_traces(opacity=1)
fig_pace.update_layout(
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    title_x=0.5,
    margin=dict(l=60, r=5, t=50, b=60),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig_pace.show()

In [13]:
fig_pace.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Team Pace Plots/{}/{} {} Sprint Race Team Pace Plot.html".format(year,year,gp),full_html=False, include_plotlyjs='cdn')

In [14]:
#Get MINIMUM OF THE MEDIAN LAP TIMES

fastest_lap = transformed_laps["median"].min()
transformed_laps['Delta']= transformed_laps["median"] - (fastest_lap)
transformed_laps['Delta_Percent']= (transformed_laps["median"]/(fastest_lap)-1)*100
transformed_laps['Delta_Percent_Abs']=transformed_laps['Delta_Percent']
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+100
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].round(3)
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].astype(str)
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+"%"
transformed_laps_relative_median=transformed_laps[["Team","Delta","Delta_Percent","Delta_Percent_Abs"]].drop_duplicates()
transformed_laps_relative_median=transformed_laps_relative_median.sort_values("Delta")
transformed_laps_relative_median

,Team,Delta,Delta_Percent,Delta_Percent_Abs
660,McLaren,0.0000,100.0%,0.000000
663,Mercedes,0.1230,100.145%,0.144813
670,Red Bull Racing,0.2555,100.301%,0.300811
675,Aston Martin,0.9220,101.086%,1.085510
680,Williams,1.0595,101.247%,1.247395
681,Racing Bulls,1.1225,101.322%,1.321568
683,Kick Sauber,1.2645,101.489%,1.488750
685,Ferrari,1.2800,101.507%,1.506999
684,Haas F1 Team,1.2840,101.512%,1.511709
691,Alpine,1.6345,101.924%,1.924367


In [15]:
for index, row in transformed_laps_relative_median.iterrows():
  if transformed_laps_relative_median.loc[index, 'Delta']==0.0:
    transformed_laps_relative_median.loc[index, 'Delta']=0.001
for index, row in transformed_laps_relative_median.iterrows():
  if transformed_laps_relative_median.loc[index, 'Delta_Percent_Abs']==0.0:
    transformed_laps_relative_median.loc[index, 'Delta_Percent_Abs']=0.001
        
transformed_laps_relative_median

,Team,Delta,Delta_Percent,Delta_Percent_Abs
660,McLaren,0.0010,100.0%,0.001000
663,Mercedes,0.1230,100.145%,0.144813
670,Red Bull Racing,0.2555,100.301%,0.300811
675,Aston Martin,0.9220,101.086%,1.085510
680,Williams,1.0595,101.247%,1.247395
681,Racing Bulls,1.1225,101.322%,1.321568
683,Kick Sauber,1.2645,101.489%,1.488750
685,Ferrari,1.2800,101.507%,1.506999
684,Haas F1 Team,1.2840,101.512%,1.511709
691,Alpine,1.6345,101.924%,1.924367


In [16]:
fig_median=px.bar(
    transformed_laps_relative_median,
    x="Team",
    y="Delta_Percent_Abs",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace (Median Lap Times) for the {} {} Sprint Race</b>".format(year,gp),
    text="Delta_Percent",
    height=700, width=1200,
color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)

fig_median.update_traces(textposition='outside')

fig_median.update_traces(opacity=0.85)

fig_median.update_traces( marker_line_color='white',marker_line_width=1.5)

fig_median.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

fig_median.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

fig_median.update_yaxes(range=[transformed_laps_relative_median['Delta_Percent_Abs'].max()+0.8, 0])

# fig['layout']['yaxis']['autorange'] = "reversed"

fig_median.update_layout(
    xaxis={'side': 'top'},
    yaxis={'side': 'left'}
)

fig_median.update_xaxes(title_text='')      # Remove axis titles
fig_median.update_yaxes(title_text='Percentage%')

fig_median.update_xaxes(
        title_standoff = 75
)

for x,y in zip(transformed_laps_relative_median.Team, transformed_laps_relative_median.Delta_Percent_Abs):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_LOGOS").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig_median.add_layout_image(
          x=x,
          y=y+0.3,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=0.9,
          sizey=0.9,
          xanchor="center",
          yanchor="middle",
      )

fig_median.update_layout(
    title_x=0.5,
    margin=dict(l=100, r=5, t=110, b=30),
    hoverlabel=dict(
    bgcolor="white",
    font_size=20,
    font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=16,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig_median.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)

fig_median.show()

In [17]:
fig_median.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Median Lap Times/{}/{} {} Sprint Race Team Median Plot.html".format(year,year,gp),full_html=False, include_plotlyjs='cdn')

In [18]:
#Get MINIMUM OF ALL LAP TIMES
transformed_laps['min_all'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('min')

fastest_lap_all = transformed_laps["min_all"].min()
transformed_laps['Delta_Min_All']= transformed_laps["min_all"] - (fastest_lap_all)
transformed_laps['Delta_Percent_All']= (transformed_laps["min_all"]/(fastest_lap_all)-1)*100
transformed_laps['Delta_Percent_Abs']=transformed_laps['Delta_Percent_All']
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+100
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].round(3)
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].astype(str)
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+"%"
transformed_laps_relative_min_all=transformed_laps[["Team","Delta_Min_All","Delta_Percent_All","Delta_Percent_Abs"]].drop_duplicates()
transformed_laps_relative_min_all=transformed_laps_relative_min_all.sort_values("Delta_Min_All")
transformed_laps_relative_min_all

,Team,Delta_Min_All,Delta_Percent_All,Delta_Percent_Abs
691,Alpine,0.000,100.0%,0.000000
675,Aston Martin,0.397,100.477%,0.477232
660,McLaren,0.800,100.962%,0.961677
663,Mercedes,1.276,101.534%,1.533875
670,Red Bull Racing,1.353,101.626%,1.626437
681,Racing Bulls,2.216,102.664%,2.663846
680,Williams,2.275,102.735%,2.734769
684,Haas F1 Team,2.367,102.845%,2.845362
685,Ferrari,2.450,102.945%,2.945136
683,Kick Sauber,2.475,102.975%,2.975189


In [19]:
for index, row in transformed_laps_relative_min_all.iterrows():
  if transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']==0.0:
    transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']=0.001
for index, row in transformed_laps_relative_min_all.iterrows():
  if transformed_laps_relative_min_all.loc[index, 'Delta_Percent_Abs']==0.0:
    transformed_laps_relative_min_all.loc[index, 'Delta_Percent_Abs']=0.001
        
transformed_laps_relative_min_all

,Team,Delta_Min_All,Delta_Percent_All,Delta_Percent_Abs
691,Alpine,0.001,100.0%,0.001000
675,Aston Martin,0.397,100.477%,0.477232
660,McLaren,0.800,100.962%,0.961677
663,Mercedes,1.276,101.534%,1.533875
670,Red Bull Racing,1.353,101.626%,1.626437
681,Racing Bulls,2.216,102.664%,2.663846
680,Williams,2.275,102.735%,2.734769
684,Haas F1 Team,2.367,102.845%,2.845362
685,Ferrari,2.450,102.945%,2.945136
683,Kick Sauber,2.475,102.975%,2.975189


In [20]:
fig_min=px.bar(
    transformed_laps_relative_min_all,
    x="Team",
    y="Delta_Percent_Abs",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace (Fastest Lap Times) for the {} {} Sprint Race</b>".format(year,gp),
    text="Delta_Percent_All",
    height=700, width=1200,
color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Racing Bulls": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)

fig_min.update_traces(textposition='outside')

fig_min.update_traces(opacity=0.85)

fig_min.update_traces( marker_line_color='white',marker_line_width=1.5)

fig_min.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

fig_min.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

fig_min.update_yaxes(range=[transformed_laps_relative_min_all['Delta_Percent_Abs'].max()+2, 0])

# fig['layout']['yaxis']['autorange'] = "reversed"

fig_min.update_layout(
    xaxis={'side': 'top'},
    yaxis={'side': 'left'}
)

fig_min.update_xaxes(title_text='')      # Remove axis titles
fig_min.update_yaxes(title_text='Percentage%')

fig_min.update_xaxes(
        title_standoff = 75
)

for x,y in zip(transformed_laps_relative_min_all.Team, transformed_laps_relative_min_all.Delta_Percent_Abs):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_LOGOS").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig_min.add_layout_image(
          x=x,
          y=y+0.8,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=0.9,
          sizey=0.9,
          xanchor="center",
          yanchor="middle",
      )


fig_min.update_layout(
    title_x=0.5,
    margin=dict(l=100, r=5, t=110, b=30),
    hoverlabel=dict(
    bgcolor="white",
    font_size=20,
    font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=16,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig_min.update_yaxes(ticksuffix = "  ")
fig_min.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)

fig_min.show()

In [21]:
fig_min.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Fastest Lap/{}/{} {} Sprint Race Team Fastest Lap Plot.html".format(year,year,gp),full_html=False, include_plotlyjs='cdn')

In [22]:
#Get AVERAGE OF ALL LAP TIMES
transformed_laps['avg_all'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform(np.mean)

fastest_lap_avg = transformed_laps["avg_all"].min()
transformed_laps['Delta_Avg_All']= transformed_laps["avg_all"] - (fastest_lap_avg)
transformed_laps['Delta_Percent_Avg']= (transformed_laps["avg_all"]/(fastest_lap_avg)-1)*100
transformed_laps['Delta_Percent_Abs']=transformed_laps['Delta_Percent_Avg']
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg']+100
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg'].round(3)
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg'].astype(str)
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg']+"%"
transformed_laps_relative_avg_all=transformed_laps[["Team","Delta_Avg_All","Delta_Percent_Avg","Delta_Percent_Abs"]].drop_duplicates()
transformed_laps_relative_avg_all=transformed_laps_relative_avg_all.sort_values("Delta_Avg_All")
transformed_laps_relative_avg_all

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_38334/970362312.py:2: FutureWarning:

The provided callable <function mean at 0x1084eb560> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.



,Team,Delta_Avg_All,Delta_Percent_Avg,Delta_Percent_Abs
660,McLaren,0.000000,100.0%,0.000000
663,Mercedes,0.299750,100.353%,0.353242
670,Red Bull Racing,0.343944,100.405%,0.405323
680,Williams,1.083000,101.276%,1.276266
681,Racing Bulls,1.273222,101.5%,1.500434
683,Kick Sauber,1.426417,101.681%,1.680966
684,Haas F1 Team,1.438750,101.696%,1.695501
675,Aston Martin,1.454173,101.714%,1.713676
685,Ferrari,1.618972,101.908%,1.907884
691,Alpine,1.636462,101.928%,1.928495


In [23]:
for index, row in transformed_laps_relative_avg_all.iterrows():
  if transformed_laps_relative_avg_all.loc[index, 'Delta_Avg_All']==0.0:
    transformed_laps_relative_avg_all.loc[index, 'Delta_Avg_All']=0.001
for index, row in transformed_laps_relative_avg_all.iterrows():
  if transformed_laps_relative_avg_all.loc[index, 'Delta_Percent_Abs']==0.0:
    transformed_laps_relative_avg_all.loc[index, 'Delta_Percent_Abs']=0.001   
        
transformed_laps_relative_avg_all

,Team,Delta_Avg_All,Delta_Percent_Avg,Delta_Percent_Abs
660,McLaren,0.001000,100.0%,0.001000
663,Mercedes,0.299750,100.353%,0.353242
670,Red Bull Racing,0.343944,100.405%,0.405323
680,Williams,1.083000,101.276%,1.276266
681,Racing Bulls,1.273222,101.5%,1.500434
683,Kick Sauber,1.426417,101.681%,1.680966
684,Haas F1 Team,1.438750,101.696%,1.695501
675,Aston Martin,1.454173,101.714%,1.713676
685,Ferrari,1.618972,101.908%,1.907884
691,Alpine,1.636462,101.928%,1.928495


In [24]:
fig=px.bar(
    transformed_laps_relative_avg_all,
    x="Team",
    y="Delta_Percent_Abs",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace (Average Lap Times) for the {} {}</b>".format(year,gp),
    text="Delta_Percent_Avg",
    height=700, width=1200,
color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Racing Bulls": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)

fig.update_traces(textposition='outside')

fig.update_traces(opacity=0.85)

fig.update_traces( marker_line_color='white',marker_line_width=1.5)

fig.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

fig.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

fig.update_yaxes(range=[transformed_laps_relative_avg_all['Delta_Percent_Abs'].max()+1, 0])

# fig['layout']['yaxis']['autorange'] = "reversed"

fig.update_layout(
    xaxis={'side': 'top'},
    yaxis={'side': 'left'}
)

fig.update_xaxes(title_text='')      # Remove axis titles
fig.update_yaxes(title_text='Percentage%')

fig.update_xaxes(
        title_standoff = 75
)

for x,y in zip(transformed_laps_relative_avg_all.Team, transformed_laps_relative_avg_all.Delta_Percent_Abs):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_LOGOS").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig.add_layout_image(
          x=x,
          y=y+0.7,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=0.9,
          sizey=0.9,
          xanchor="center",
          yanchor="middle",
      )

fig.update_layout(
    title_x=0.5,
    margin=dict(l=100, r=5, t=110, b=30),
    hoverlabel=dict(
    bgcolor="white",
    font_size=20,
    font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=16,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)

fig.show()

In [25]:
fig.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Average Lap Times/{}/{} {} Driver Average Lap Plot.html".format(year, year,gp),full_html=False, include_plotlyjs='cdn')

In [ ]:
#Multi GP Plot

In [32]:
#2025
race_multi=['Miami Grand Prix','Belgian Grand Prix','São Paulo Grand Prix','United States Grand Prix']
#2024
# race_multi=['Austrian Grand Prix','Miami Grand Prix','Chinese Grand Prix']
#2023
# race_multi=['Austrian Grand Prix','Azerbaijan Grand Prix','Belgian Grand Prix','Qatar Grand Prix','São Paulo Grand Prix','United States Grand Prix']
#2022
# race_multi=['Austrian Grand Prix','Emilia Romagna Grand Prix','São Paulo Grand Prix']
#2021
# race_multi=['British Grand Prix','Italian Grand Prix','São Paulo Grand Prix']

year_multi=2025
for i in race_multi:
    track2=track3[(track3["GP"]==i)&(track3["Year"]==year_multi)]
    year=track2['Year'].iloc[0]
    gp=track2['GP'].iloc[0]
    track2['LapTime']= pd.to_timedelta(track2["LapTime"])
    track2=track2[track2['LapNumber']!=1.0]
    quicklaps=track2[track2["LapTime"]<track2['LapTime'].min()*1.07]
    transformed_laps = quicklaps.copy()
    transformed_laps.loc[:, "LapTime (s)"] = quicklaps["LapTime"].dt.total_seconds()

    # order the team from the fastest (lowest median lap time) tp slower
    team_order = (
        transformed_laps[["Team", "LapTime (s)"]].groupby("Team").median()["LapTime (s)"].sort_values().index
    )
    print(team_order)
    transformed_laps['median'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('median')
    transformed_laps.tail()
    fig_pace=px.box(
        transformed_laps.sort_values(by=["median","LapNumber"]),
        x="Team",
        y="LapTime (s)",
        color='Team',
        template="xgridoff",
        title="<b>Team Pace Plot for the {} {} Sprint Race</b>".format(year_multi,i),
        height=700, width=1200,

    color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "Alfa Romeo Racing":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
            }
    )
    fig_pace.update_traces(opacity=1)
    fig_pace.update_layout(
        title_x=0.5,
        hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
        ),
        yaxis = dict(tickfont = dict(size=15)),
        xaxis = dict(tickfont = dict(size=15)),
        font=dict(
            family="PT Sans Narrow",
            size=14,
            color="Black"
        ),
        title_font_family="PT Sans Narrow",
        margin=dict(l=50, r=5, t=35, b=50)
    )
    fig_pace.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Team Pace Plots/{}/{} {} Sprint Race Team Pace Plot.html".format(year_multi,year_multi,i),full_html=False, include_plotlyjs='cdn')
    #Get MINIMUM OF THE MEDIAN LAP TIMES

    fastest_lap = transformed_laps["median"].min()
    transformed_laps['Delta']= transformed_laps["median"] - (fastest_lap)
    transformed_laps['Delta_Percent']= (transformed_laps["median"]/(fastest_lap)-1)*100
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+100
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].round(3)
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].astype(str)
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+"%"
    transformed_laps_relative_median=transformed_laps[["Team","Delta","Delta_Percent"]].drop_duplicates()
    transformed_laps_relative_median=transformed_laps_relative_median.sort_values("Delta")
    transformed_laps_relative_median
    for index, row in transformed_laps_relative_median.iterrows():
        if transformed_laps_relative_median.loc[index, 'Delta']==0.0:
         transformed_laps_relative_median.loc[index, 'Delta']=0.001
    transformed_laps_relative_median
    fig_median=px.bar(
        transformed_laps_relative_median,
        x="Team",
        y="Delta",
        color='Team',
        template="xgridoff",
        title="<b>Team Pace (Median Lap Times) for the {} {} Sprint Race</b>".format(year_multi,i),
        text="Delta_Percent",
        height=700, 
        width=1200,
        color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "Alfa Romeo Racing":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
            }
    )

    fig_median.update_traces(textposition='outside')

    fig_median.update_traces(opacity=0.85)

    fig_median.update_traces( marker_line_color='white',marker_line_width=1.5)

    fig_median.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_median.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_median.update_yaxes(range=[transformed_laps_relative_median['Delta'].max()+1.5, 0])

    fig_median.update_layout(
        xaxis={'side': 'top'},
        yaxis={'side': 'left'}
    )

    fig_median.update_xaxes(title_text='')      # Remove axis titles
    fig_median.update_yaxes(title_text='')

    fig_median.update_xaxes(
            title_standoff = 75
    )

    fig_median.update_layout(
        title_x=0.5,
        margin=dict(l=60, r=5, t=110, b=30),
        hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
        ),
        yaxis = dict(tickfont = dict(size=15)),
        xaxis = dict(tickfont = dict(size=15)),
        font=dict(
            family="PT Sans Narrow",
            size=12,
            color="Black"
        ),
        title_font_family="PT Sans Narrow"
    )
    fig_median.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)
    fig_median.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Median Lap Times/{}/{} {} Sprint Race Team Median Plot.html".format(year_multi,year_multi,i),full_html=False, include_plotlyjs='cdn')
    
  #Get MINIMUM OF ALL LAP TIMES
    transformed_laps['min_all'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('min')
    fastest_lap_all = transformed_laps["min_all"].min()
    transformed_laps['Delta_Min_All']= transformed_laps["min_all"] - (fastest_lap_all)
    transformed_laps['Delta_Percent_All']= (transformed_laps["min_all"]/(fastest_lap_all)-1)*100
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+100
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].round(3)
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].astype(str)
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+"%"
    transformed_laps_relative_min_all=transformed_laps[["Team","Delta_Min_All","Delta_Percent_All"]].drop_duplicates()
    transformed_laps_relative_min_all=transformed_laps_relative_min_all.sort_values("Delta_Min_All")
    transformed_laps_relative_min_all
    for index, row in transformed_laps_relative_min_all.iterrows():
        if transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']==0.0:
            transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']=0.001
    fig_min=px.bar(
        transformed_laps_relative_min_all,
        x="Team",
        y="Delta_Min_All",
        color='Team',
        template="xgridoff",
        title="<b>Team Pace (Fastest Lap Times) for the {} {} Sprint Race</b>".format(year,gp),
        text="Delta_Percent_All",
        height=700, width=1200,
    color_discrete_map={
                     "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "Alfa Romeo Racing":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                    }
    )

    fig_min.update_traces(textposition='outside')

    fig_min.update_traces(opacity=0.85)

    fig_min.update_traces( marker_line_color='white',marker_line_width=1.5)

    fig_min.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_min.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_min.update_yaxes(range=[transformed_laps_relative_min_all['Delta_Min_All'].max()+1.5, 0])

    # fig['layout']['yaxis']['autorange'] = "reversed"

    fig_min.update_layout(
        xaxis={'side': 'top'},
        yaxis={'side': 'left'}
    )

    fig_min.update_xaxes(title_text='')      # Remove axis titles
    fig_min.update_yaxes(title_text='')

    fig_min.update_xaxes(
            title_standoff = 75
    )

    fig_min.update_layout(
        title_x=0.5,
        margin=dict(l=60, r=5, t=110, b=30),
        hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
        ),
        yaxis = dict(tickfont = dict(size=15)),
        xaxis = dict(tickfont = dict(size=15)),
        font=dict(
            family="PT Sans Narrow",
            size=14,
            color="Black"
        ),
        title_font_family="PT Sans Narrow"
    )
    fig_min.update_yaxes(ticksuffix = "  ")
    fig_min.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)
    fig_min.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Fastest Lap/{}/{} {} Sprint Race Team Fastest Lap Plot.html".format(year_multi,year_multi,i),full_html=False, include_plotlyjs='cdn')

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_58120/1965670226.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_58120/1965670226.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_58120/1965670226.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Index(['McLaren', 'Red Bull Racing', 'Mercedes', 'Ferrari', 'Aston Martin',
       'Racing Bulls', 'Williams', 'Alpine', 'Haas F1 Team', 'Kick Sauber'],
      dtype='object', name='Team')
Index(['McLaren', 'Red Bull Racing', 'Haas F1 Team', 'Williams', 'Ferrari',
       'Racing Bulls', 'Kick Sauber', 'Aston Martin', 'Mercedes', 'Alpine'],
      dtype='object', name='Team')
Index(['McLaren', 'Mercedes', 'Ferrari', 'Aston Martin', 'Alpine',
       'Red Bull Racing', 'Kick Sauber', 'Haas F1 Team', 'Williams',
       'Racing Bulls'],
      dtype='object', name='Team')
Index(['Ferrari', 'Red Bull Racing', 'Williams', 'Mercedes', 'Racing Bulls',
       'Haas F1 Team', 'Aston Martin', 'Kick Sauber', 'Alpine'],
      dtype='object', name='Team')


/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_58120/1965670226.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

